In [ ]:
# =============================================================================
# @title VISTA Framework — Multi-Model Divergence-Aware Stock Price Forecasting
# (Hardened: minimax-m3 + nemotron + ministral-14b cross-model consensus,
#            VIX daily contrarian chart, TV-regularized derivative overlay,
#            sliding-window RPM rate limiter, Retry-After-aware backoff,
#            Parquet-cached yfinance, Numba-JIT CPU kernels,
#            free-tier-40-RPM-safe concurrency)
# =============================================================================
# Models     : MiniMax-M3 (minimaxai/minimax-m3) — 428B MoE VLM, ≈22B active
#              NVIDIA Nemotron 3 Nano Omni 30B A3B (Reasoning)
#              Ministral 14B (Mistral)
# Provider   : NVIDIA NIM free endpoint (https://integrate.api.nvidia.com/v1)
# Environment: Google Colab (CPU, 2 cores) + Numba JIT
# Reference  : https://arxiv.org/abs/2505.18570
#
# Description:
# An end-to-end stock price forecasting pipeline that runs on Google Colab
# (CPU-only) and queries THREE Vision-Language Models in parallel via the free
# NVIDIA NIM API endpoint. For each ticker, the script downloads three
# timeframes of market data (weekly, daily, 15-minute) using yfinance
# (Parquet-cached), computes Numba-accelerated technical indicators
# (Volume-Weighted Moving Average, log returns, momentum z-score) and a
# Total-Variation-regularized derivative (TV Derivative) trend overlay, and
# renders Heikin-Ashi candlestick charts with VWMA + TV Derivative overlays —
# stripped of all text labels, ticker identifiers, and price annotations to
# prevent the VLM from relying on prior knowledge. A fourth chart (daily VIX)
# provides equity-market volatility context as a contrarian signal.
#
# MULTI-MODEL CONSENSUS (generalizes single-model N-ensemble median):
#   * Three models (MiniMax-M3, Nemotron, Ministral-14B) are each called
#     ONCE per ticker (3 VLM calls total — same count as the original
#     N_ENSEMBLE=3 design, but now 3 DISTINCT architectures for true
#     cross-model diversity instead of 3 same-model samples).
#   * Per-day consensus (generalized to N models):
#       - If (max(predictions[i]) - min(predictions[i])) / close
#         <= DIVERGENCE_TOLERANCE: consensus[i] = arithmetic mean (agreement)
#       - Else: consensus[i] = median([*predictions[i], anchor[i]])
#         where anchor[i] = close * exp((i+1) * mean(r5)) — a random-walk-
#         with-drift stabilizer that prevents hallucination-driven blowups.
#       With N=3 models + anchor, the median of 4 values is the average of
#       the two middle values, which is more robust than a simple mean.
#   * disagreement_pct (mean per-day relative spread) is surfaced as a
#     confidence signal in the output table and chart.
#   * Graceful degradation: if only 1 model succeeds, its forecast is used
#     with a "degraded" confidence flag. If >=2 succeed, full consensus runs.
#
# VIX CONTRARIAN SIGNAL (NEW):
#   * CBOE Volatility Index (^VIX) is downloaded ONCE per pipeline run
#     (global signal — same for every ticker) and rendered as a 4th chart.
#   * VIX captures equity-market risk expectations orthogonal to any single
#     ticker's price history. Lower VIX → bullish bias; higher VIX → bearish.
#   * For crypto / non-equity tickers the VIX is a weak signal; the prompt
#     instructs the model to weight it only for equity tickers.
#
# TV DERIVATIVE OVERLAY (NEW):
#   * Total-Variation-regularized derivative (deeptime.util.diff.tv_derivative)
#     overlaid as a yellow line on each price chart — represents the
#     noise-robust trend slope. Positive slope = uptrend, negative = downtrend,
#     zero crossing = potential reversal. Values are NOT shown (shape only).
#   * Computed on tail(CHART_BARS) only (n=100) → ~12ms per call.
#   * Graceful fallback to np.gradient() if deeptime is unavailable or the
#     TV solver fails to converge.
#
# RATE-LIMIT HARDENING (NVIDIA NIM free tier — global 40 RPM hard cap):
#   * Thread-safe sliding-window RateLimiter (TARGET_RPM = 30) enforces a
#     25% safety margin below the 40 RPM sandbox ceiling.
#   * VLM_CONCURRENCY=3 — matches the tri-model call shape (one in-flight
#     request per model).
#   * call_vlm_with_retry: up to 5 exponential-backoff retries, honoring
#     HTTP `Retry-After` headers with full jitter.
#
# YFINANCE HARDENING:
#   * Parquet on-disk cache (~/.vista_cache/) with TTL = CACHE_TTL_HOURS.
#   * Exponential backoff: base_sleep * 2^attempt, max RETRY_MAX attempts.
#   * threads=True delegated to yf.download (no manual batch threading).
#   * repair=False for ALL intervals — yfinance 1.4.x crashes with
#     KeyError('Stock Splits') when repair=True on many tickers (crypto
#     pairs, Brazilian FIIs, even AAPL in some environments).
# =============================================================================

import os
import sys
import io
import time
import json
import math
import base64
import random
import re
import logging
import threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from statistics import median, mean

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib
matplotlib.use("Agg")  # Thread-safe background rendering
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numba
from openai import OpenAI

# ── TV Derivative (deeptime) — optional dependency with fallback ────────────
# deeptime.util.diff.tv_derivative provides noise-robust trend slope via
# Total-Variation regularization. If unavailable, _compute_tv_derivative
# falls back to numpy.gradient (less robust but always present).
try:
    from deeptime.util import diff as _deeptime_diff
    _HAS_DEEPTIME = True
except ImportError:
    _HAS_DEEPTIME = False

# ── IPython / Colab display support ─────────────────────────────────────────
try:
    from IPython.display import display as ipy_display, Image as ipy_Image
    _HAS_IPY = True
except ImportError:
    _HAS_IPY = False


# ── API key resolution (Colab Secrets first, then env var) ──────────────────
def _get_secret_api_key(name: str) -> str:
    """Fetch API key from Colab Secrets (userdata) or OS environment."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)


_GET_API_KEY_NVIDIA = lambda: _get_secret_api_key("NVIDIA_API_KEY")

# ── Tickers to forecast (space-separated, edit in-line) ─────────────────────
TICKERS_INPUT = (
    "BTC-USD SOL-USD XRP-USD QDFI11.SA URPR11.SA VSLH11.SA BYND CHTR PATH PYPL UPST AMAT MU STX"
)

# ── Model registry — NVIDIA NIM free endpoint ───────────────────────────────
# Three primary models are used in parallel for divergence-aware consensus:
#   * minimax-m3     — 428B MoE VLM (≈22B active), broad-coverage chart reading
#   * nemotron       — 30B reasoning model, strong on cross-timeframe alignment
#   * ministral-14b  — 14B instruct VLM (FP8), fast, strong vision/instruction
#                      adherence, optimized for constrained environments.
MODEL_REGISTRY = {
    "minimax-m3": {
        "label":         "MiniMax-M3 (minimaxai/minimax-m3)",
        "model_id":      "minimaxai/minimax-m3",
        "temperature":   0.25,
        "top_p":         1.00,
        "max_tokens":    4096,
        "extra_body":    None,
        "system_prompt": "You are a JSON-only forecasting engine. Output only a 5-element JSON array of closing prices.",
    },
    "nemotron": {
        "label":         "Nemotron 3 Nano Omni 30B A3B (Reasoning)",
        "model_id":      "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning",
        "temperature":   0.25,
        "top_p":         1.00,
        "max_tokens":    8192,
        "extra_body": {
            "chat_template_kwargs": {"enable_thinking": True},
            "reasoning_budget":     2048,
        },
        "system_prompt": "You are a JSON-only forecasting engine. Output only a 5-element JSON array of closing prices.",
    },
    "ministral-14b": {
        "label":         "Ministral 3 14B Instruct 2512 FP8 (mistralai/ministral-14b-instruct-2512)",
        "model_id":      "mistralai/ministral-14b-instruct-2512",
        "temperature":   0.15,  # Recommended by model card for best results
        "top_p":         1.00,
        "max_tokens":    4096,
        "extra_body":    None,
        "system_prompt": "You are a JSON-only forecasting engine. Output only a 5-element JSON array of closing prices.",
    },
}

# Primary multi-model set — order does not matter (parallel dispatch).
# All models are called once per ticker; consensus is computed via
# _divergence_aware_consensus which generalizes to N models.
VLM_PRIMARY_MODELS = ("minimax-m3", "nemotron", "ministral-14b")

# ── Pipeline knobs ──────────────────────────────────────────────────────────
PERIOD_DAILY       = "2y"
PERIOD_WEEKLY      = "2y"
PERIOD_15M         = "60d"
VWMA_WINDOW        = 20
MOM_PERIOD         = 14
MOM_ZSCORE_WINDOW  = 60
CHART_BARS         = 100
IMAGE_DPI          = 150
TIMEOUT_VLM        = 180
OUTPUT_DIR         = "/content" if os.path.exists("/content") else "."
LOG_LEVEL          = logging.INFO

# ── yfinance rules (per yf_rules in PRD) ────────────────────────────────────
RETRY_LIMIT        = 3        # yfinance download retries
BASE_SLEEP         = 1        # base_sleep for exponential backoff
WORKER_POOL        = 8        # informational; concurrency delegated to yf.download(threads=True)
CHUNK_SIZE         = 100_000  # array segment size for chunked dispatch
CACHE_TTL_HOURS    = 12       # Parquet cache time-to-live
CACHE_DIR          = os.path.join(os.path.expanduser("~"), ".vista_cache")

# ── Multi-model consensus knobs ─────────────────────────────────────────────
# Per-day relative divergence threshold below which the model forecasts are
# averaged. Above it, a trend-extrapolated anchor is added and the median
# of (predictions + anchor) is taken — preventing hallucination-driven blowups.
# The threshold is relative to the last close, so it scales with price level.
DIVERGENCE_TOLERANCE = 0.015  # 1.5% per-day relative spread

# ── VIX contrarian signal knobs ─────────────────────────────────────────────
# CBOE Volatility Index ticker on yfinance. Downloaded ONCE per pipeline run
# (global signal — same for every ticker) and rendered as a 4th chart.
VIX_TICKER          = "^VIX"
VIX_PERIOD          = "2y"    # daily VIX history period
VIX_HIGH_THRESHOLD  = 25.0    # above this → high fear, reversal risk
VIX_LOW_THRESHOLD   = 15.0    # below this → complacency, trend continuation

# ── TV Derivative knobs ─────────────────────────────────────────────────────
# Total-Variation regularization strength. Smaller alpha = smoother derivative
# (more regularization); larger alpha = closer to raw finite differences.
# 0.001 is a good default for price data in the $10–$1000 range.
TV_ALPHA            = 0.001
TV_TOL              = 1e-5
TV_FD_WINDOW_RADIUS = 5
TV_SPARSE           = False   # False is faster for n <= ~500

# ── Concurrency knobs ───────────────────────────────────────────────────────
# VLM_CONCURRENCY=3 matches the tri-model call shape exactly (one in-flight
# request per model). Ticker pipelines are intentionally limited to keep
# the global RPM well below the 30 RPM ceiling.
VLM_CONCURRENCY         = 3
TICKER_PIPELINE_WORKERS = 4

# ── Rate-limit knobs (NVIDIA NIM free tier — 40 RPM hard cap) ───────────────
TARGET_RPM              = 30
RATE_LIMIT_WINDOW_SEC   = 60.0

VLM_RETRY_MAX           = 5
VLM_RETRY_BASE_DELAY    = 5.0
VLM_RETRY_MAX_DELAY     = 60.0

# ── Thread-safety primitives ────────────────────────────────────────────────
DOWNLOAD_LOCK  = threading.Lock()
VLM_SEMAPHORE  = threading.Semaphore(VLM_CONCURRENCY)


# =============================================================================
# Sliding-window rate limiter (thread-safe)
# =============================================================================

class _SlidingWindowRateLimiter:
    """Thread-safe sliding-window rate limiter enforcing a hard RPM ceiling.

    Tracks request timestamps in a fixed-width window (default 60s). When the
    window is full, ``acquire()`` blocks the calling thread until enough
    timestamps age out. This guarantees the global request rate never exceeds
    ``max_rpm`` even when many ensemble workers and ticker pipelines run
    concurrently — the key safeguard against NVIDIA NIM free-tier 429 errors.
    """

    def __init__(self, max_rpm: int, window_sec: float = RATE_LIMIT_WINDOW_SEC):
        """Configure the limiter with an RPM ceiling and window length."""
        if max_rpm <= 0:
            raise ValueError("max_rpm must be > 0")
        self.max_rpm     = max_rpm
        self.window_sec  = float(window_sec)
        self._timestamps = deque()
        self._lock       = threading.Lock()
        self._min_spacing = self.window_sec / float(max_rpm)

    def _try_acquire_locked(self) -> tuple:
        """Attempt one acquisition under lock; return (acquired, sleep_for_sec).

        Caller must hold ``self._lock``. On success, records a timestamp and
        returns ``(True, 0.0)``. On failure, returns ``(False, wait_seconds)``
        so the caller can sleep and retry.
        """
        now = time.monotonic()
        cutoff = now - self.window_sec
        while self._timestamps and self._timestamps[0] <= cutoff:
            self._timestamps.popleft()

        if len(self._timestamps) >= self.max_rpm:
            return False, self._timestamps[0] + self.window_sec - now

        sleep_for = 0.0
        if self._timestamps:
            elapsed = now - self._timestamps[-1]
            if elapsed < self._min_spacing:
                sleep_for = self._min_spacing - elapsed
        if sleep_for <= 0:
            self._timestamps.append(time.monotonic())
            return True, 0.0
        return False, sleep_for

    def acquire(self) -> None:
        """Block until a request slot is available, then record it."""
        while True:
            with self._lock:
                acquired, sleep_for = self._try_acquire_locked()
            if acquired:
                return
            if sleep_for > 0:
                time.sleep(min(sleep_for, self.window_sec))

    def stats(self) -> dict:
        """Snapshot of current limiter state (for logging/debugging)."""
        with self._lock:
            now = time.monotonic()
            cutoff = now - self.window_sec
            while self._timestamps and self._timestamps[0] <= cutoff:
                self._timestamps.popleft()
            return {
                "in_window":  len(self._timestamps),
                "max_rpm":    self.max_rpm,
                "window_sec": self.window_sec,
            }


VLM_RATE_LIMITER = _SlidingWindowRateLimiter(max_rpm=TARGET_RPM)

# ── Logging ─────────────────────────────────────────────────────────────────
log = logging.getLogger("VISTA")
log.setLevel(LOG_LEVEL)
if log.hasHandlers():
    log.handlers.clear()
_log_handler = logging.StreamHandler(sys.stdout)
_log_handler.setLevel(LOG_LEVEL)
_log_handler.setFormatter(logging.Formatter(
    "%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S"))
log.addHandler(_log_handler)
log.propagate = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)


# ── PROMPT template ─────────────────────────────────────────────────────────
PROMPT = """
You are a quantitative forecasting analyst. Predict the next 5 trading days' closing prices (D+1 to D+5) for an anonymous synthetic price series.

INPUTS:
1. Four charts (ignore all text, labels, or tickers on every chart):
   a. Heikin-Ashi candlestick chart with VWMA overlay (Blue line) and TV Derivative overlay (Yellow line) — WEEKLY timeframe (Left).
   b. Heikin-Ashi candlestick chart with VWMA overlay (Blue line) and TV Derivative overlay (Yellow line) — DAILY timeframe (Center).
   c. Heikin-Ashi candlestick chart with VWMA overlay (Blue line) and TV Derivative overlay (Yellow line) — 15-MINUTE timeframe (Right).
   d. Daily chart of the CBOE Volatility Index (VIX) — a contrarian equity-market risk signal (Bottom).
2. Pre-computed data:
   - Last close: {close}
   - 5-day log returns: [{r1}, {r2}, {r3}, {r4}, {r5}]
   - Price vs VWMA distance: {dist}%
   - Momentum z-score: {z}
   - VIX last value: {vix_close}
   - VIX 5-day change: {vix_change}%

INDICATOR DEFINITIONS:
- VWMA (Blue line): Volume-Weighted Moving Average. Price above VWMA = bullish bias; below = bearish; crossing = transition.
- TV Derivative (Yellow line): Total-Variation-regularized derivative of the close price. Represents the noise-robust trend SLOPE — values are intentionally hidden, only the shape matters. Yellow line above zero = uptrend; below zero = downtrend; crossing zero = potential reversal. Steeper slope = stronger trend. Divergence between price and TV Derivative (e.g. price making new highs while TV Derivative slopes down) signals momentum exhaustion.
- VIX (Bottom chart): CBOE Volatility Index — equity-market's 30-day forward volatility expectation. INVERSE correlation with equity prices: lower VIX → higher probability of higher prices; higher VIX → higher probability of lower prices. VIX > {vix_high} = high fear (bearish pressure, reversal risk); VIX < {vix_low} = complacency (bullish, trend continuation). For non-equity tickers (crypto, commodities) VIX is a weak signal — weight it lightly.

ANALYSIS INSTRUCTIONS (If you are a reasoning model, use your internal thought process here. Do not output your analysis in the final text):
1. Trend Classification: For each timeframe, classify the trend (up, down, sideways, reversal) based on Heikin-Ashi rules (consecutive same-color candles = trend; shrinking bodies = exhaustion; crossing VWMA = transition) AND the TV Derivative slope sign and steepness.
2. Cross-Timeframe Alignment: Rank confidence. High if all 3 agree, moderate if 2 agree, low/volatile if all differ.
3. Data Cross-Check: Do the 5-day log returns match the visual trend? Does |dist|% suggest continuation (<3%) or mean reversion (>5%)? Is |z| > 2 (extreme, reversal risk) or < 1 (normal)?
4. VIX Cross-Check: Is VIX trending up (bearish pressure) or down (bullish pressure)? Is VIX > {vix_high} (high fear) or < {vix_low} (complacency)? Apply VIX as a contrarian bias ONLY for equity tickers; for crypto/commodities, ignore VIX.
5. TV Derivative Cross-Check: On each timeframe, is the yellow line above or below zero? Is it diverging from price? A zero-crossing in the most recent bars signals a potential reversal.
6. Trajectory Formation: Anchor the trajectory to the last close ({close}). Keep daily step sizes typically 0.1%–2%. Keep the 5-day range within ±2% to ±15% of the last close, scaled by perceived volatility. If a major reversal is signaled, ensure a smooth transition, not an erratic jump.

OUTPUT FORMAT (CRITICAL):
Your final response must contain NOTHING EXCEPT a JSON array of 5 chronological closing prices rounded to two decimal places.
Do not include any text, explanations, markdown formatting (like ```json), or apologies.

Correct example: [285.50, 287.20, 286.80, 289.10, 291.50]
"""


# =============================================================================
# Numba JIT functions (nesting depth strictly < 2; nogil=True; NO prange,
# NO parallel=True — those silently produce wrong results on Colab CPU)
# =============================================================================

@numba.jit(nopython=True, nogil=True, cache=True)
def _init_nan(n: int) -> np.ndarray:
    """Return a length-n float32 array filled with NaN."""
    out = np.empty(n, dtype=np.float32)
    for i in range(n):
        out[i] = np.nan
    return out


@numba.jit(nopython=True, nogil=True, cache=True)
def _vwma_step(close, volume, window, i, csum, vsum):
    """Advance the VWMA rolling sum by one bar; return (csum, vsum, val)."""
    csum += close[i] * volume[i]
    vsum += volume[i]
    if i >= window:
        csum -= close[i - window] * volume[i - window]
        vsum -= volume[i - window]
    val = np.float32(np.nan)
    if i >= window - 1 and vsum != np.float32(0.0):
        val = csum / vsum
    return csum, vsum, val


@numba.jit(nopython=True, nogil=True, cache=True)
def _vwma_loop(close, volume, window, out):
    """Compute Volume-Weighted Moving Average over a 1-D close/volume pair."""
    csum = np.float32(0.0)
    vsum = np.float32(0.0)
    n = close.shape[0]
    for i in range(n):
        csum, vsum, val = _vwma_step(close, volume, window, i, csum, vsum)
        out[i] = val
    return out


@numba.jit(nopython=True, nogil=True, cache=True)
def _log_returns_last_n(close, n: int) -> np.ndarray:
    """Return the last n log returns as a float32 array."""
    out = np.empty(n, dtype=np.float32)
    s = close.shape[0] - n
    for i in range(n):
        out[i] = np.log(close[s + i] / close[s + i - 1])
    return out


@numba.jit(nopython=True, nogil=True, cache=True)
def _rolling_momentum(close, period: int, out_len: int) -> np.ndarray:
    """Compute rolling momentum (close[i] - close[i-period]) for the last out_len bars."""
    out = np.empty(out_len, dtype=np.float32)
    start = close.shape[0] - out_len
    for i in range(out_len):
        out[i] = close[start + i] - close[start + i - period]
    return out


@numba.jit(nopython=True, nogil=True, cache=True)
def _ha_open_loop(o, h, l, c, ha_o):
    """Compute Heikin-Ashi open prices.

    Formula: ha_o[0] = (o[0] + c[0]) / 2
             ha_o[i] = (ha_o[i-1] + ha_c[i-1]) / 2
             where ha_c[i-1] = (o[i-1] + h[i-1] + l[i-1] + c[i-1]) / 4

    Sequential dependency prevents vectorization; Numba JIT with nogil=True
    is the fastest path on Colab CPU.
    """
    n = o.shape[0]
    if n == 0:
        return ha_o
    ha_o[0] = (o[0] + c[0]) / 2.0
    for i in range(1, n):
        ha_c_prev = (o[i - 1] + h[i - 1] + l[i - 1] + c[i - 1]) / 4.0
        ha_o[i] = (ha_o[i - 1] + ha_c_prev) / 2.0
    return ha_o


# =============================================================================
# Data helpers — yfinance download with Parquet caching & exponential backoff
# =============================================================================

def to_f32_1d(s) -> np.ndarray:
    """Coerce a pandas Series to a contiguous float32 1-D numpy array."""
    return np.ascontiguousarray(np.ravel(s.to_numpy(dtype=np.float32)))


def _find_level(columns, ticker: str) -> int:
    """Return the MultiIndex level (if any) on which ``ticker`` appears."""
    for lvl in range(columns.nlevels):
        if ticker in columns.levels[lvl]:
            return lvl
    return -1


def _handle_multiindex(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Project a MultiIndex DataFrame down to the ticker's slice."""
    level = _find_level(df.columns, ticker)
    if level != -1:
        return df.xs(ticker, axis=1, level=level)
    df.columns = df.columns.get_level_values(0)
    return df


def _sanitize_df_columns(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Normalize column names: collapse MultiIndex, deduplicate columns."""
    if df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df = _handle_multiindex(df, ticker)
    if not isinstance(df.columns, pd.MultiIndex):
        df.columns = [str(col) for col in df.columns]
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()]
    return df


def _get_download_kwargs(interval: str, period: str, auto_adjust: bool = True) -> dict:
    """Build yf.download kwargs; intraday uses start/end, others use period.

    ``repair=False`` is set for ALL intervals. yfinance's ``repair=True``
    triggers a ``KeyError('Stock Splits')`` bug on weekly/daily downloads
    for many tickers (crypto pairs, Brazilian FIIs, and even regular stocks
    like AAPL in some environments). The repair feature attempts to fix
    100x-unit price spikes but is not essential for our use case — we
    already validate interval spacing in ``validate_and_fix_interval``.

    ``auto_adjust`` is parameterized for the two-pass strategy in
    ``download_ticker``: try True first (fast path), fall back to
    False + ``_manual_adjust`` if the fast path fails.
    """
    kwargs = {
        "interval":          interval,
        "threads":           True,
        "progress":          False,
        "auto_adjust":       auto_adjust,
        "repair":            False,
        "multi_level_index": False,
    }
    if interval in ("15m", "5m", "1h"):
        match = re.match(r"(\d+)d", period)
        days = min(int(match.group(1)), 59) if match else 59
        end_dt = datetime.now()
        start_dt = end_dt - timedelta(days=days)
        kwargs["start"] = start_dt.strftime("%Y-%m-%d")
        kwargs["end"]   = (end_dt + timedelta(days=1)).strftime("%Y-%m-%d")
        return kwargs
    kwargs["period"] = period
    return kwargs


def _execute_download(ticker: str, kwargs: dict) -> pd.DataFrame:
    """Call yf.download under DOWNLOAD_LOCK (yfinance is not fully thread-safe)."""
    with DOWNLOAD_LOCK:
        return yf.download(ticker, **kwargs)


def _manual_adjust(df: pd.DataFrame) -> pd.DataFrame:
    """Apply price adjustment manually using the ``Adj Close`` column.

    Replaces yfinance's ``auto_adjust=True`` path, which crashes with
    ``KeyError('Stock Splits')`` on tickers that lack stock-split data
    (crypto pairs, Brazilian FIIs, etc.). Multiplies OHLC by the
    ``Adj Close / Close`` ratio and drops auxiliary columns.
    """
    if df.empty:
        return df
    if "Adj Close" not in df.columns:
        drop_cols = [c for c in ("Dividends", "Stock Splits") if c in df.columns]
        return df.drop(columns=drop_cols) if drop_cols else df
    close = df["Close"].replace(0, np.nan)
    adj_ratio = (df["Adj Close"] / close).fillna(1.0)
    for col in ("Open", "High", "Low", "Close"):
        if col in df.columns:
            df[col] = df[col] * adj_ratio
    drop_cols = [c for c in ("Adj Close", "Dividends", "Stock Splits") if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df


def _try_download_attempt(ticker: str, interval: str, kwargs: dict,
                          manual_adjust: bool = False) -> pd.DataFrame:
    """Single download attempt; raises on empty result or interval mismatch.

    When ``manual_adjust=True``, applies ``_manual_adjust`` after sanitization
    (used for the ``auto_adjust=False`` fallback path).
    """
    df = _execute_download(ticker, kwargs)
    if df.empty:
        raise ValueError(f"Empty dataframe returned for {ticker} {interval}")
    df = _sanitize_df_columns(df, ticker)
    if manual_adjust:
        df = _manual_adjust(df)
    return validate_and_fix_interval(df, interval, ticker)


def _handle_download_error(ticker: str, interval: str, attempt: int, exc: Exception) -> None:
    """Log a failed download attempt and sleep with exponential backoff."""
    log.warning("Download attempt %d/%d failed for %s %s: %s",
                attempt + 1, RETRY_LIMIT, ticker, interval, exc)
    time.sleep(BASE_SLEEP * (2 ** attempt))


def _attempt_download_step(ticker: str, interval: str, kwargs: dict, attempt: int,
                           manual_adjust: bool = False):
    """Wrap a single attempt; returns None on failure after backoff."""
    try:
        return _try_download_attempt(ticker, interval, kwargs, manual_adjust=manual_adjust)
    except Exception as exc:
        _handle_download_error(ticker, interval, attempt, exc)
        return None


def _cache_path(ticker: str, interval: str) -> str:
    """Return the Parquet cache path for a (ticker, interval) pair."""
    safe_ticker = re.sub(r"[^A-Za-z0-9._-]", "_", ticker)
    return os.path.join(CACHE_DIR, f"{safe_ticker}_{interval}.parquet")


def _try_load_parquet_cache(ticker: str, interval: str) -> pd.DataFrame:
    """Return cached DataFrame if fresh (< CACHE_TTL_HOURS old), else empty."""
    path = _cache_path(ticker, interval)
    if not os.path.exists(path):
        return pd.DataFrame()
    age_hours = (time.time() - os.path.getmtime(path)) / 3600.0
    if age_hours >= CACHE_TTL_HOURS:
        return pd.DataFrame()
    try:
        df = pd.read_parquet(path)
        log.debug("Cache hit for %s %s (age=%.1fh)", ticker, interval, age_hours)
        return df
    except Exception as exc:
        log.debug("Parquet cache read failed for %s %s: %s", ticker, interval, exc)
        return pd.DataFrame()


def _persist_parquet_cache(ticker: str, interval: str, df: pd.DataFrame) -> None:
    """Persist a downloaded DataFrame to Parquet for future cache hits."""
    if df.empty:
        return
    path = _cache_path(ticker, interval)
    try:
        df.to_parquet(path)
        log.debug("Parquet cache written for %s %s", ticker, interval)
    except Exception as exc:
        log.debug("Parquet cache write failed for %s %s: %s", ticker, interval, exc)


def download_ticker(ticker: str, interval: str, period: str) -> pd.DataFrame:
    """Download a single (ticker, interval) with Parquet cache + retries.

    Cache lookup is O(1) on disk; a fresh hit avoids the yfinance HTTP call
    entirely. On cache miss, falls back to yf.download with exponential
    backoff (base_sleep * 2^attempt) up to RETRY_MAX attempts.

    Two-pass strategy to dodge yfinance's ``KeyError('Stock Splits')`` bug:
      Pass 1 (auto_adjust=True): the fast path; works for most tickers.
      Pass 2 (auto_adjust=False + _manual_adjust): fallback for tickers
        whose data lacks the Stock Splits column (crypto pairs, Brazilian
        FIIs, etc.) — yfinance's internal auto-adjust crashes on those.
    """
    cached = _try_load_parquet_cache(ticker, interval)
    if not cached.empty:
        return cached

    # Pass 1: auto_adjust=True (the original behavior).
    kwargs_adj = _get_download_kwargs(interval, period, auto_adjust=True)
    for attempt in range(RETRY_LIMIT):
        res = _attempt_download_step(ticker, interval, kwargs_adj, attempt)
        if res is not None:
            _persist_parquet_cache(ticker, interval, res)
            return res

    # Pass 2: auto_adjust=False + manual adjustment (fallback).
    log.info("Pass 1 (auto_adjust=True) exhausted for %s %s; "
             "falling back to auto_adjust=False + manual adjust.",
             ticker, interval)
    kwargs_raw = _get_download_kwargs(interval, period, auto_adjust=False)
    for attempt in range(RETRY_LIMIT):
        res = _attempt_download_step(ticker, interval, kwargs_raw, attempt,
                                     manual_adjust=True)
        if res is not None:
            _persist_parquet_cache(ticker, interval, res)
            return res
    raise RuntimeError(
        f"Download failed after {RETRY_LIMIT * 2} retries (both passes): "
        f"{ticker} {interval}"
    )


def _retry_download_fixed(ticker: str, interval: str) -> pd.DataFrame:
    """Fallback intraday retry with explicit start/end bounds and threads=False.

    Uses ``auto_adjust=False`` + manual adjustment to dodge the
    ``KeyError('Stock Splits')`` bug (see ``_get_download_kwargs``).
    """
    end_dt = datetime.now()
    start_dt = end_dt - timedelta(days=59)
    with DOWNLOAD_LOCK:
        df = yf.download(
            ticker,
            interval=interval,
            start=start_dt.strftime("%Y-%m-%d"),
            end=(end_dt + timedelta(days=1)).strftime("%Y-%m-%d"),
            threads=False,
            progress=False,
            auto_adjust=False,
            repair=False,
            multi_level_index=False,
        )
    if not df.empty:
        df = _manual_adjust(df)
    return df


def _check_and_apply_retry(ticker: str, interval: str, expected_sec: dict):
    """Apply the fixed-bounds retry; return repaired df or None."""
    df2 = _retry_download_fixed(ticker, interval)
    if df2.empty:
        return None
    df2 = _sanitize_df_columns(df2, ticker)
    # _manual_adjust already applied inside _retry_download_fixed.
    deltas2 = pd.Series(df2.index).diff().dropna()
    median_sec2 = deltas2.median().total_seconds()
    if median_sec2 <= expected_sec[interval] * 10:
        log.debug("Retry succeeded for %s %s", ticker, interval)
        return df2
    return None


def validate_and_fix_interval(df: pd.DataFrame, interval: str, ticker: str) -> pd.DataFrame:
    """For intraday frames, verify bar spacing matches the expected interval.

    If the median delta exceeds 10× the expected interval, attempt one
    fixed-bounds retry. Raises if the retry also fails.
    """
    if interval in ("1wk", "1d"):
        return df
    expected_sec = {"15m": 15 * 60, "5m": 5 * 60, "1h": 3600}
    if interval not in expected_sec:
        return df
    deltas = pd.Series(df.index).diff().dropna()
    median_sec = deltas.median().total_seconds()
    log.debug("Interval check for %s %s: median_delta=%.0fs, expected=%ds",
              ticker, interval, median_sec, expected_sec[interval])
    if median_sec <= expected_sec[interval] * 10:
        return df
    log.warning("Interval mismatch for %s %s: median_delta=%.0fs. Retrying...",
                ticker, interval, median_sec)
    df2 = _check_and_apply_retry(ticker, interval, expected_sec)
    if df2 is not None:
        return df2
    raise ValueError(
        f"Could not obtain valid {interval} data for {ticker}. Last bar spacing: {median_sec}s"
    )


def heikin_ashi(df: pd.DataFrame) -> pd.DataFrame:
    """Convert OHLCV to Heikin-Ashi OHLC (float32 outputs).

    Uses _ha_open_loop (Numba JIT) for the sequential ha_o recurrence;
    ha_c, ha_h, ha_l are computed in vectorized numpy.
    """
    o = df["Open"].to_numpy(dtype=np.float64).ravel()
    h = df["High"].to_numpy(dtype=np.float64).ravel()
    l = df["Low"].to_numpy(dtype=np.float64).ravel()
    c = df["Close"].to_numpy(dtype=np.float64).ravel()
    n = len(c)

    ha_c = (o + h + l + c) / 4.0
    ha_o = np.empty(n, dtype=np.float64)
    _ha_open_loop(o, h, l, c, ha_o)
    ha_h = np.maximum(h, np.maximum(ha_o, ha_c))
    ha_l = np.minimum(l, np.minimum(ha_o, ha_c))

    out = pd.DataFrame(index=df.index)
    out["Open"]  = ha_o.astype(np.float32)
    out["High"]  = ha_h.astype(np.float32)
    out["Low"]   = ha_l.astype(np.float32)
    out["Close"] = ha_c.astype(np.float32)
    return out


# =============================================================================
# TV Derivative — Total-Variation regularized trend slope (optional deeptime)
# =============================================================================

def _compute_tv_derivative(close: np.ndarray) -> np.ndarray:
    """Compute the TV-regularized derivative of a 1-D close-price array.

    Uses deeptime.util.diff.tv_derivative when available (~12ms for n=100);
    falls back to np.gradient (always available, less noise-robust) otherwise.
    Also falls back if the TV solver fails to converge on pathological inputs
    (all-constant series, NaN-laden data, etc.).

    Args:
        close: 1-D numpy array of close prices (any float dtype).

    Returns:
        1-D float32 array of derivative values, same length as ``close``.
    """
    n = close.shape[0]
    if n < 2:
        return np.zeros(n, dtype=np.float32)
    y = np.asarray(close, dtype=np.float64)
    x = np.arange(n, dtype=np.float64)
    if _HAS_DEEPTIME:
        try:
            df_tv = _deeptime_diff.tv_derivative(
                x, y, alpha=TV_ALPHA, tol=TV_TOL,
                fd_window_radius=TV_FD_WINDOW_RADIUS, sparse=TV_SPARSE,
            )
            return np.asarray(df_tv, dtype=np.float32)
        except Exception as exc:
            log.debug("TV derivative failed (%s); falling back to np.gradient.", exc)
    # Fallback: simple finite differences (less robust, always available).
    return np.gradient(y, x).astype(np.float32)


# =============================================================================
# VIX — CBOE Volatility Index (downloaded once per pipeline run)
# =============================================================================

# Module-level cache for the VIX DataFrame + rendered chart path.
# VIX is a global signal (same for every ticker), so we download it once
# per pipeline run and reuse the chart PNG across all tickers.
_VIX_CACHE_LOCK  = threading.Lock()
_VIX_DF_CACHE    = None     # pd.DataFrame or None
_VIX_CHART_PATH  = None     # str or None


def download_vix() -> pd.DataFrame:
    """Download the daily VIX series ONCE per pipeline run (thread-safe).

    Returns the cached DataFrame on subsequent calls. VIX has no Volume
    column — the download path handles this gracefully via _manual_adjust.
    """
    global _VIX_DF_CACHE
    with _VIX_CACHE_LOCK:
        if _VIX_DF_CACHE is not None:
            return _VIX_DF_CACHE
    df = download_ticker(VIX_TICKER, "1d", VIX_PERIOD)
    with _VIX_CACHE_LOCK:
        _VIX_DF_CACHE = df
    return df


def _vix_trend_label(last: float, change_pct: float) -> str:
    """Classify VIX trend from last value + 5-day change.

    Priority: high-fear > complacency > rising > falling.
    """
    if last >= VIX_HIGH_THRESHOLD:
        return "high-fear"
    if last <= VIX_LOW_THRESHOLD:
        return "complacency"
    if change_pct > 0:
        return "rising"
    return "falling"


def _compute_vix_metrics(vix_df: pd.DataFrame) -> dict:
    """Compute VIX last value + 5-day percentage change.

    Returns a dict with keys 'vix_close', 'vix_change', 'vix_trend'.
    On any error returns zeros so the prompt formatting never crashes.
    """
    if vix_df is None or vix_df.empty:
        return {"vix_close": 0.0, "vix_change": 0.0, "vix_trend": "n/a"}
    try:
        closes = vix_df["Close"].to_numpy(dtype=np.float64)
        if len(closes) < 6:
            return {"vix_close": float(closes[-1]), "vix_change": 0.0,
                    "vix_trend": "n/a"}
        last  = float(closes[-1])
        prev5 = float(closes[-6])
        change_pct = 100.0 * ((last / prev5) - 1.0) if prev5 > 0 else 0.0
        trend = _vix_trend_label(last, change_pct)
        return {"vix_close": round(last, 2),
                "vix_change": round(change_pct, 2),
                "vix_trend": trend}
    except Exception as exc:
        log.debug("VIX metrics computation failed: %s", exc)
        return {"vix_close": 0.0, "vix_change": 0.0, "vix_trend": "n/a"}


def _plot_vix_chart(vix_df: pd.DataFrame, path: str,
                    dpi: int = IMAGE_DPI) -> None:
    """Render the daily VIX line chart to PNG (text-label-free, dark theme).

    Draws the last CHART_BARS daily VIX closes as a simple line. No candles
    (VIX is an index, not OHLC-tradable). Colored zones mark high-fear (red)
    and complacency (green) thresholds for visual context.
    """
    vix_slice = vix_df.tail(CHART_BARS)
    closes = vix_slice["Close"].to_numpy(dtype=np.float32)
    n = len(closes)
    x = np.arange(n, dtype=np.float32)

    fig, ax = plt.subplots(figsize=(8, 3), dpi=dpi, facecolor="#0d1117")
    ax.set_facecolor("#0d1117")

    # Threshold bands (subtle, no text labels)
    ax.axhline(y=VIX_HIGH_THRESHOLD, color="#f85149", linewidth=0.6,
               alpha=0.4, linestyle="--")
    ax.axhline(y=VIX_LOW_THRESHOLD, color="#3fb950", linewidth=0.6,
               alpha=0.4, linestyle="--")

    ax.plot(x, closes, color="#ffa657", linewidth=1.5, alpha=0.95)
    ax.set_xlim(-1, n)
    y_min = float(np.nanmin(closes)) * 0.95
    y_max = float(np.nanmax(closes)) * 1.05
    ax.set_ylim(y_min, y_max)
    ax.set_axis_off()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0,
                facecolor=fig.get_facecolor())
    plt.close(fig)
    png_bytes = buf.getvalue()
    if not png_bytes:
        raise FileNotFoundError(f"Failed to generate VIX chart image: {path}")
    with open(path, "wb") as f:
        f.write(png_bytes)


def get_vix_chart_path() -> str | None:
    """Return the path to the shared VIX chart PNG, downloading + rendering once.

    Returns None if VIX download fails (the pipeline continues without the
    VIX chart — the VLM gets 3 charts instead of 4).
    """
    global _VIX_CHART_PATH
    with _VIX_CACHE_LOCK:
        if _VIX_CHART_PATH is not None and os.path.exists(_VIX_CHART_PATH):
            return _VIX_CHART_PATH
    try:
        vix_df = download_vix()
        path = os.path.join(OUTPUT_DIR, "vista_vix_daily.png")
        _plot_vix_chart(vix_df, path)
        with _VIX_CACHE_LOCK:
            _VIX_CHART_PATH = path
        log.info("VIX chart rendered once (shared across all tickers).")
        return path
    except Exception as exc:
        log.warning("VIX chart unavailable — continuing without it: %s", exc)
        return None


def reset_vix_cache() -> None:
    """Clear the module-level VIX cache (for testing / re-runs)."""
    global _VIX_DF_CACHE, _VIX_CHART_PATH
    with _VIX_CACHE_LOCK:
        _VIX_DF_CACHE = None
        _VIX_CHART_PATH = None


def _draw_tv_overlay(ax, x: np.ndarray, tv_line, n: int) -> None:
    """Draw the TV Derivative yellow line on a secondary y-axis (no values).

    Called from _plot_single_ha only when tv_line is provided. The secondary
    axis scale is independent of the price axis — only the sign (above/below
    zero) and zero-crossings matter, not the magnitude.
    """
    tv = np.asarray(tv_line, dtype=np.float32).ravel()
    if tv.shape[0] != n:
        return
    ax2 = ax.twinx()
    ax2.set_facecolor("none")  # transparent — shows price axis below
    ax2.plot(x, tv, color="#ffdf5d", linewidth=1.3, alpha=0.85)
    ax2.axhline(y=0.0, color="#ffdf5d", linewidth=0.4, alpha=0.3, linestyle=":")
    ax2.set_xlim(-1, n)
    tv_finite = tv[np.isfinite(tv)]
    if tv_finite.shape[0] > 0:
        tv_pad = (tv_finite.max() - tv_finite.min()) * 0.1
        ax2.set_ylim(float(tv_finite.min()) - tv_pad,
                     float(tv_finite.max()) + tv_pad)
    ax2.set_axis_off()


# =============================================================================
# Chart generation — saves silently to disk
# =============================================================================

def _plot_single_ha(df_ha: pd.DataFrame, vwma_line, path: str,
                    tv_line=None, dpi: int = IMAGE_DPI) -> None:
    """Render a single Heikin-Ashi + VWMA + optional TV Derivative chart to PNG.

    The TV Derivative overlay (yellow line) is drawn on a secondary y-axis
    so its scale (trend slope) doesn't distort the price axis. Values are
    NOT shown — only the line shape, per the PRD requirement.
    """
    from matplotlib.collections import LineCollection, PolyCollection

    o = df_ha["Open"].to_numpy(dtype=np.float32).ravel()
    h = df_ha["High"].to_numpy(dtype=np.float32).ravel()
    l = df_ha["Low"].to_numpy(dtype=np.float32).ravel()
    c = df_ha["Close"].to_numpy(dtype=np.float32).ravel()
    n = len(c)
    x = np.arange(n, dtype=np.float32)
    vv = np.asarray(vwma_line, dtype=np.float32).ravel()

    fig, ax = plt.subplots(figsize=(8, 5), dpi=dpi, facecolor="#0d1117")
    ax.set_facecolor("#0d1117")

    # Vectorized wick segments: shape (n, 2, 2). Avoids n Python list
    # allocations — LineCollection accepts numpy arrays natively.
    wick_segs = np.empty((n, 2, 2), dtype=np.float32)
    wick_segs[:, 0, 0] = x
    wick_segs[:, 0, 1] = l
    wick_segs[:, 1, 0] = x
    wick_segs[:, 1, 1] = h
    wick_col  = LineCollection(wick_segs, colors="#8b949e", linewidths=0.5)
    ax.add_collection(wick_col)

    lower  = np.minimum(o, c)
    height = np.abs(c - o)
    height = np.where(height < 1e-6, 0.1, height)
    colors = np.where(c >= o, "#3fb950", "#f85149")

    # Vectorized candle bodies via PolyCollection — accepts a single (n, 4, 2)
    # array of polygon vertices, avoiding n Python Rectangle patch objects.
    body_verts = np.empty((n, 4, 2), dtype=np.float32)
    body_verts[:, 0, 0] = x - 0.3
    body_verts[:, 0, 1] = lower
    body_verts[:, 1, 0] = x + 0.3
    body_verts[:, 1, 1] = lower
    body_verts[:, 2, 0] = x + 0.3
    body_verts[:, 2, 1] = lower + height
    body_verts[:, 3, 0] = x - 0.3
    body_verts[:, 3, 1] = lower + height
    body_col = PolyCollection(body_verts, facecolors=colors,
                              edgecolors=colors, linewidths=0, antialiaseds=False)
    ax.add_collection(body_col)

    ax.plot(x, vv, color="#58a6ff", linewidth=1.2, alpha=0.85)

    # TV Derivative overlay on secondary y-axis (yellow line, no values).
    # Drawn AFTER VWMA so it sits on top visually. Delegated to a helper
    # to keep nesting depth ≤ 2.
    if tv_line is not None:
        _draw_tv_overlay(ax, x, tv_line, n)

    ax.set_xlim(-1, n)
    ax.set_ylim(float(np.nanmin(l)) * 0.995, float(np.nanmax(h)) * 1.005)
    ax.set_axis_off()
    # Save via BytesIO buffer — single-shot disk write is faster than
    # savefig's incremental file streaming. Also lets us verify size in-memory.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0,
                facecolor=fig.get_facecolor())
    plt.close(fig)
    png_bytes = buf.getvalue()
    if not png_bytes:
        raise FileNotFoundError(f"Failed to generate chart image: {path}")
    with open(path, "wb") as f:
        f.write(png_bytes)


def generate_chart_images(weekly, daily, intraday, vw_w, vw_d, vw_i,
                         ticker_tag: str, tv_w=None, tv_d=None, tv_i=None,
                         vix_chart_path: str = None) -> list:
    """Render the three-timeframe chart triplet + optional VIX chart.

    Args:
        weekly, daily, intraday: OHLCV DataFrames (full downloaded history).
        vw_w, vw_d, vw_i: VWMA arrays aligned to the full history.
        ticker_tag: tag used in the output filename.
        tv_w, tv_d, tv_i: optional TV Derivative arrays for the tail(CHART_BARS)
            slice. If None, no TV overlay is drawn.
        vix_chart_path: optional pre-rendered VIX chart PNG path. If provided,
            it is appended to the returned paths list as the 4th chart.

    Returns:
        List of PNG paths (3 or 4 elements depending on VIX availability).
    """
    w  = weekly.tail(CHART_BARS)
    d  = daily.tail(CHART_BARS)
    i_ = intraday.tail(CHART_BARS)

    w_ha = heikin_ashi(w)
    d_ha = heikin_ashi(d)
    i_ha = heikin_ashi(i_)

    paths = []
    frames = [
        ("wk", w_ha, vw_w[-CHART_BARS:], tv_w),
        ("dy", d_ha, vw_d[-CHART_BARS:], tv_d),
        ("15", i_ha, vw_i[-CHART_BARS:], tv_i),
    ]
    for label, df_ha, vw, tv in frames:
        p = os.path.join(OUTPUT_DIR, f"vista_{ticker_tag}_{label}.png")
        _plot_single_ha(df_ha, vw, p, tv_line=tv)
        paths.append(p)
    if vix_chart_path is not None and os.path.exists(vix_chart_path):
        paths.append(vix_chart_path)
    return paths


# =============================================================================
# Metrics computation
# =============================================================================

def compute_metrics(dc, dv, vix_metrics: dict = None) -> dict:
    """Compute VWMA distance, last-5 log returns, momentum z-score, and VIX context.

    Args:
        dc: daily close array (float32).
        dv: daily volume array (float32).
        vix_metrics: optional dict from _compute_vix_metrics. If None, VIX
            fields default to zeros.
    """
    vw_d = _vwma_loop(dc, dv, VWMA_WINDOW, _init_nan(dc.shape[0]))
    r5 = _log_returns_last_n(dc, 5)
    dist = (dc[-1] / vw_d[-1] - 1.0) * 100.0

    lookback = min(MOM_ZSCORE_WINDOW, len(dc) - MOM_PERIOD)
    if lookback < 2:
        z = 0.0
    else:
        mom_series = _rolling_momentum(dc, MOM_PERIOD, lookback)
        mu    = np.mean(mom_series)
        sigma = np.std(mom_series)
        z = (mom_series[-1] - mu) / sigma if sigma > 1e-8 else 0.0

    if vix_metrics is None:
        vix_metrics = {"vix_close": 0.0, "vix_change": 0.0, "vix_trend": "n/a"}

    return {
        "close":      float(dc[-1]),
        "r1":         round(float(r5[0]), 4),
        "r2":         round(float(r5[1]), 4),
        "r3":         round(float(r5[2]), 4),
        "r4":         round(float(r5[3]), 4),
        "r5":         round(float(r5[4]), 4),
        "dist":       round(float(dist), 2),
        "z":          round(float(z), 2),
        "vix_close":   vix_metrics["vix_close"],
        "vix_change":  vix_metrics["vix_change"],
        "vix_trend":   vix_metrics["vix_trend"],
        "vix_high":    VIX_HIGH_THRESHOLD,
        "vix_low":     VIX_LOW_THRESHOLD,
    }


# =============================================================================
# VLM parsing and inference pipeline
# =============================================================================

def _get_ignore_returns(metrics) -> list:
    """Return the r1..r5 list from metrics, or None on failure."""
    if metrics is None:
        return None
    try:
        return [
            float(metrics["r1"]),
            float(metrics["r2"]),
            float(metrics["r3"]),
            float(metrics["r4"]),
            float(metrics["r5"]),
        ]
    except (KeyError, TypeError, ValueError):
        return None


def _is_echo_returns(parsed_vals: list, ignore_returns: list) -> bool:
    """Detect when the VLM echoes the input returns instead of producing prices."""
    if ignore_returns is None:
        return False
    return all(abs(p - r) < 1e-4 for p, r in zip(parsed_vals, ignore_returns))


def _is_flat_line(vals: list) -> bool:
    """Detect flat-line or near-flat-line hallucinations."""
    if len(vals) < 2:
        return False
    # Calculate mean and standard deviation
    mu = sum(vals) / len(vals)
    if abs(mu) < 1e-8:
        return True  # All values are essentially zero
    variance = sum((x - mu) ** 2 for x in vals) / len(vals)
    std_dev = math.sqrt(variance)
    # If standard deviation is less than 0.01% of the mean, it's a flat line
    return (std_dev / abs(mu)) < 1e-4


def _try_parse_match(match_str: str, ignore_returns: list):
    """Parse a bracketed JSON array of 5 finite floats; reject echo/flat-line."""
    try:
        arr = json.loads(match_str)
        if not isinstance(arr, list) or len(arr) != 5:
            return None
        parsed_vals = [float(x) for x in arr]
        # Reject NaN / Inf — they would silently propagate as broken forecasts
        if not all(math.isfinite(v) for v in parsed_vals):
            return None
        if _is_echo_returns(parsed_vals, ignore_returns):
            return None
        # Reject flat-line hallucinations (e.g., [100.0, 100.0, 100.0, 100.0, 100.0])
        if _is_flat_line(parsed_vals):
            log.debug("Rejected flat-line forecast: %s", parsed_vals)
            return None
        return parsed_vals
    except (json.JSONDecodeError, ValueError, TypeError):
        return None


def _search_bracket_arrays(text_clean: str, ignore_returns: list):
    """Scan for the first valid 5-element bracketed array in ``text_clean``."""
    matches = [m.group() for m in re.finditer(r"\[[^\]]*\]", text_clean)]
    for m in matches:
        parsed = _try_parse_match(m, ignore_returns)
        if parsed is not None:
            return parsed
    return None


def _apply_returns_to_close(parsed_array: list, close: float) -> list:
    """Convert a parsed returns array into cumulative-multiply prices."""
    prices = []
    curr = close
    for r in parsed_array:
        curr = curr * np.exp(r)
        prices.append(round(float(curr), 2))
    return prices


def _convert_returns_to_prices_if_needed(parsed_array: list, close: float) -> list:
    """If parsed values look like log returns (all in [-1.5, 1.5]) and close>5,
    auto-convert them to absolute prices via cumulative multiplication.
    """
    if not all(-1.5 <= v <= 1.5 for v in parsed_array) or close <= 5.0:
        return parsed_array
    log.warning("Forecast values %s detected as returns. Auto-converting...", parsed_array)
    return _apply_returns_to_close(parsed_array, close)


def parse_vlm_array(text: str, close: float, metrics: dict = None) -> list:
    """Parse a VLM response into a 5-element list of absolute closing prices."""
    if text is None:
        raise ValueError("VLM returned None content")
    text_str   = str(text)
    text_clean = re.sub(r"```(?:json)?", "", text_str).strip()
    ignore_returns = _get_ignore_returns(metrics)

    parsed_array = _search_bracket_arrays(text_clean, ignore_returns)
    if parsed_array is None:
        parsed_array = _try_parse_match(text_clean, ignore_returns)

    if parsed_array is None:
        raise ValueError(f"No valid 5-element JSON array found in VLM response: {text[:300]}")

    return _convert_returns_to_prices_if_needed(parsed_array, close)


def _get_vlm_params(model_key: str) -> dict:
    """Resolve VLM parameters + API key for a model registry entry."""
    if model_key not in MODEL_REGISTRY:
        raise ValueError(
            f"Unknown VLM model '{model_key}'. "
            f"Valid options: {', '.join(MODEL_REGISTRY.keys())}"
        )
    api_key = _GET_API_KEY_NVIDIA()
    if not api_key:
        raise RuntimeError(
            "CRITICAL: NVIDIA_API_KEY is not set. Add it to Colab Secrets or env."
        )
    cfg = MODEL_REGISTRY[model_key]
    return {
        "api_key":        api_key,
        "base_url":       "https://integrate.api.nvidia.com/v1",
        "model_id":       cfg["model_id"],
        "label":          cfg["label"],
        "temperature":    cfg["temperature"],
        "top_p":          cfg["top_p"],
        "max_tokens":     cfg["max_tokens"],
        "extra_body":     cfg["extra_body"],
        "system_prompt":  cfg.get("system_prompt"),
    }


def _build_vlm_content(metrics: dict) -> list:
    """Format the PROMPT with metrics (including VIX) and return the text content block."""
    prompt_text = PROMPT.format(
        close=metrics["close"],
        r1=metrics["r1"],
        r2=metrics["r2"],
        r3=metrics["r3"],
        r4=metrics["r4"],
        r5=metrics["r5"],
        dist=metrics["dist"],
        z=metrics["z"],
        vix_close=metrics.get("vix_close", 0.0),
        vix_change=metrics.get("vix_change", 0.0),
        vix_high=metrics.get("vix_high", VIX_HIGH_THRESHOLD),
        vix_low=metrics.get("vix_low", VIX_LOW_THRESHOLD),
    )
    return [{"type": "text", "text": prompt_text}]


def _append_single_image(content: list, path: str) -> None:
    """Append a single base64-encoded PNG image_url block to ``content``."""
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    content.append({
        "type":      "image_url",
        "image_url": {"url": f"data:image/png;base64,{b64}"},
    })


def _append_images_to_content(content: list, img_paths: list) -> None:
    """Append all chart PNG paths to the VLM content list."""
    for p in img_paths:
        _append_single_image(content, p)


def _create_completion(client, params: dict, content: list):
    """Build the streaming chat.completions.create call."""
    messages = []
    system_prompt = params.get("system_prompt")
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": content})

    kwargs = {
        "model":       params["model_id"],
        "messages":    messages,
        "temperature": params["temperature"],
        "top_p":       params["top_p"],
        "max_tokens":  params["max_tokens"],
        "stream":      True,
    }
    if params["extra_body"] is not None:
        kwargs["extra_body"] = params["extra_body"]
    return client.chat.completions.create(**kwargs)


def _parse_stream_chunk(chunk, reasoning_chunks: list, content_chunks: list) -> None:
    """Extract reasoning_content and content deltas from a streaming chunk."""
    if not chunk.choices:
        return
    delta = chunk.choices[0].delta
    r = getattr(delta, "reasoning_content", None)
    if r:
        reasoning_chunks.append(r)
    c = getattr(delta, "content", None)
    if c is not None:
        content_chunks.append(c)


def _consume_stream(completion):
    """Drain a streaming completion into (reasoning_str, content_str)."""
    reasoning_chunks = []
    content_chunks   = []
    for chunk in completion:
        _parse_stream_chunk(chunk, reasoning_chunks, content_chunks)
    return "".join(reasoning_chunks), "".join(content_chunks)


def _parse_reasoning_fallback(full_reasoning: str, metrics: dict, original_exc: Exception) -> list:
    """Fall back to parsing the reasoning_content if the main content failed."""
    try:
        result = parse_vlm_array(full_reasoning, metrics["close"], metrics)
        log.info("Successfully recovered forecast array from reasoning content.")
        return result
    except Exception:
        raise original_exc


def _parse_raw_response(raw: str, full_reasoning: str, metrics: dict) -> list:
    """Try parsing the main content; on failure, attempt the reasoning stream."""
    try:
        return parse_vlm_array(raw, metrics["close"], metrics)
    except Exception as exc:
        log.warning("Primary parse failed (%s). Attempting fallback parse on reasoning...", exc)
        return _parse_reasoning_fallback(full_reasoning, metrics, exc)


def call_vlm(img_paths: list, metrics: dict, model_key: str = None) -> list:
    """Send prompt + 3 images to VLM via NVIDIA NIM (single attempt, no retry).

    Acquires a token from the global sliding-window RateLimiter BEFORE issuing
    the HTTP request, so the global outbound rate never exceeds TARGET_RPM
    (default 30, safely below the NIM free-tier 40 RPM hard cap).
    """
    if model_key is None:
        model_key = VLM_PRIMARY_MODELS[0]

    params = _get_vlm_params(model_key)
    client = OpenAI(base_url=params["base_url"], api_key=params["api_key"])

    content = _build_vlm_content(metrics)
    _append_images_to_content(content, img_paths)

    log.debug("Initiating VLM stream via NVIDIA NIM for %s...", params["model_id"])

    VLM_RATE_LIMITER.acquire()
    with VLM_SEMAPHORE:
        completion     = _create_completion(client, params, content)
        full_reasoning, full_content = _consume_stream(completion)

    raw = full_content.strip() or full_reasoning.strip()
    return _parse_raw_response(raw, full_reasoning, metrics)


def _extract_retry_after(exc: Exception):
    """Return the Retry-After seconds hinted by the server, if any."""
    response = getattr(exc, "response", None)
    if response is None:
        return None
    headers = getattr(response, "headers", None) or {}
    retry_after = headers.get("Retry-After") or headers.get("retry-after")
    if not retry_after:
        return None
    try:
        return max(0.5, float(retry_after))
    except (TypeError, ValueError):
        return None


def _handle_vlm_retry_exception(exc: Exception, attempt: int, max_retries: int) -> float:
    """Inspect a VLM exception; return sleep seconds if retryable, else re-raise.

    Honors the server's ``Retry-After`` header when present; otherwise uses
    exponential backoff with full jitter.
    """
    err = str(exc)
    is_rate_limit = (
        "429" in err
        or "Too Many Requests" in err
        or "RateLimitError" in type(exc).__name__
        or "rate limit" in err.lower()
    )
    if not is_rate_limit or attempt >= max_retries - 1:
        raise exc
    server_hint = _extract_retry_after(exc)
    backoff = VLM_RETRY_BASE_DELAY * (2 ** attempt)
    backoff = min(backoff, VLM_RETRY_MAX_DELAY)
    jitter  = random.uniform(0, 1)
    wait    = max(backoff, server_hint or 0.0) + jitter
    log.warning(
        "VLM rate-limited (attempt %d/%d). Retrying in %.1fs "
        "(backoff=%.1fs, server_retry_after=%s).",
        attempt + 1, max_retries, wait, backoff, server_hint,
    )
    return wait


def call_vlm_with_retry(img_paths: list, metrics: dict,
                        model_key: str = None, max_retries: int = VLM_RETRY_MAX) -> list:
    """Call call_vlm with exponential-backoff retry on HTTP 429 rate-limit errors.

    Backoff schedule (full jitter):
        delay = min(VLM_RETRY_MAX_DELAY, base * 2^attempt) + uniform(0, 1)
    If the server supplies a Retry-After header, the larger of (backoff,
    Retry-After) is used — respecting the server's hint is the most reliable
    way to clear a 429 condition on the NIM free tier.
    """
    last_exc = None
    for attempt in range(max_retries):
        try:
            return call_vlm(img_paths, metrics, model_key)
        except Exception as e:
            last_exc = e
            wait = _handle_vlm_retry_exception(e, attempt, max_retries)
            time.sleep(wait)
    raise last_exc  # unreachable but safe


# =============================================================================
# Multi-model divergence-aware consensus (replaces ensemble_vlm)
# =============================================================================

def _compute_trend_anchor(close: float, r5_list: list, dist: float) -> list:
    """Mean-reverting random-walk-with-drift anchor.

    Blends recent momentum (mean of r5) with a mean-reversion pull toward
    the VWMA (using the distance percentage).
    If price is above VWMA (dist > 0), the anchor angles downward.
    If price is below VWMA (dist < 0), the anchor angles upward.
    """
    if not r5_list or len(r5_list) != 5 or close <= 0:
        return [round(close, 2)] * 5

    # Convert distance percentage to a decimal fraction
    dist_frac = float(dist) / 100.0

    # Blend: 50% recent drift + 50% mean reversion pull
    # If dist is positive (price > vwma), reversion is negative (pulling down)
    # If dist is negative (price < vwma), reversion is positive (pulling up)
    adjusted_drift = (float(mean(r5_list)) * 0.5) - (dist_frac * 0.5)

    return [round(close * math.exp((i + 1) * adjusted_drift), 2) for i in range(5)]


def _per_day_consensus(predictions: list, anchor: float, close: float,
                        tolerance: float) -> tuple:
    """Compute per-day consensus from N model predictions + anchor.

    Args:
        predictions: list of N float values (one per model) for a single day.
        anchor: trend-extrapolated anchor price for this day.
        close: last close price (used for relative spread calculation).
        tolerance: max relative spread below which the mean is used.

    Returns (consensus_value, is_divergent).
    """
    if not predictions:
        return round(anchor, 2), True
    if len(predictions) == 1:
        return round(float(predictions[0]), 2), False
    spread = (max(predictions) - min(predictions)) / close
    if spread <= tolerance:
        return round(sum(predictions) / len(predictions), 2), False
    return round(float(median([*predictions, anchor])), 2), True


def _divergence_aware_consensus(model_forecasts: list, metrics: dict) -> tuple:
    """Generalized divergence-aware consensus for N >= 2 models.

    Args:
        model_forecasts: list of N 5-element lists (one forecast per model).
        metrics: dict with 'close' and 'r1'..'r5' for the trend anchor.

    Per-day decision:
        - If relative spread <= DIVERGENCE_TOLERANCE: consensus = mean
        - Else: consensus = median([*predictions, anchor])

    Returns (consensus_list, disagreement_pct, confidence_label).
    """
    close = float(metrics["close"])
    r5_list = [metrics["r1"], metrics["r2"], metrics["r3"],
               metrics["r4"], metrics["r5"]]
    anchor = _compute_trend_anchor(close, r5_list, metrics["dist"])

    consensus = []
    divergent_days = 0
    total_spread = 0.0
    for i in range(5):
        day_preds = [float(m[i]) for m in model_forecasts]
        val, is_divergent = _per_day_consensus(day_preds, anchor[i],
                                                close, DIVERGENCE_TOLERANCE)
        consensus.append(val)
        if is_divergent:
            divergent_days += 1
        if len(day_preds) >= 2:
            total_spread += (max(day_preds) - min(day_preds)) / close

    disagreement_pct = round((total_spread / 5.0) * 100.0, 3)
    if divergent_days == 0:
        confidence = "high"
    elif divergent_days <= 2:
        confidence = "medium"
    else:
        confidence = "low"
    return consensus, disagreement_pct, confidence


def _collect_multi_model_results(futures) -> tuple:
    """Drain multi-model futures into (results, errors) lists.

    Returns (results, errors); order is not preserved — the consensus step
    is symmetric over the model forecasts so order does not matter.
    """
    results = []
    errors  = []
    for f in as_completed(futures):
        try:
            results.append(f.result())
        except Exception as exc:
            errors.append(str(exc))
            log.warning("Multi-model VLM call failed: %s", exc)
    return results, errors


def multi_model_consensus(img_paths: list, metrics: dict) -> tuple:
    """Call all VLM_PRIMARY_MODELS in parallel; aggregate via divergence-aware consensus.

    Returns (consensus_list, disagreement_pct, confidence_label).

    Graceful degradation:
        - N models succeed: full consensus via _divergence_aware_consensus.
        - 2..N-1 models succeed: consensus on the surviving subset.
        - 1 model succeeds: use it verbatim with confidence="degraded".
        - 0 models succeed: raise RuntimeError.
    """
    n_models = len(VLM_PRIMARY_MODELS)
    with ThreadPoolExecutor(max_workers=n_models) as pool:
        futures = [
            pool.submit(call_vlm_with_retry, img_paths, metrics, model_key)
            for model_key in VLM_PRIMARY_MODELS
        ]
        results, errors = _collect_multi_model_results(futures)

    if not results:
        error_summary = "; ".join(errors[:n_models])
        raise RuntimeError(
            f"CRITICAL: All {n_models} multi-model VLM calls failed. "
            f"Errors: {error_summary}"
        )

    if len(results) == 1:
        log.warning("Only 1 of %d models produced a forecast — using it "
                    "without consensus.", n_models)
        return results[0], 0.0, "degraded"

    if len(results) < n_models:
        log.warning("Only %d of %d models produced forecasts — running "
                    "consensus on the surviving subset.", len(results), n_models)

    return _divergence_aware_consensus(results, metrics)


# =============================================================================
# Forecast validation
# =============================================================================

def _is_finite_number(v) -> bool:
    """Return True if v is a finite float/int (not None, NaN, or Inf)."""
    if v is None:
        return False
    try:
        return math.isfinite(float(v))
    except (TypeError, ValueError):
        return False


def validate_forecast(forecast: list, close: float, ticker: str,
                      max_total_pct: float = 20.0) -> list:
    """Log a WARNING if any D+N forecast deviates > max_total_pct from last close.

    Does NOT modify the forecast — it only flags suspicious results so the
    operator can inspect them.  Returns the forecast unchanged.
    """
    if not forecast or not _is_finite_number(close) or close <= 0:
        return forecast
    for i, price in enumerate(forecast):
        if not _is_finite_number(price):
            continue
        pct = abs(float(price) / close - 1.0) * 100.0
        if pct > max_total_pct:
            log.warning(
                "[%s] Forecast D+%d=%.2f deviates %.1f%% from last close=%.2f — "
                "likely VLM hallucination. Treat this ticker's output with caution.",
                ticker, i + 1, price, pct, close)
            break
    return forecast


# =============================================================================
# Per-ticker pipeline
# =============================================================================

def _download_all_timeframes(ticker: str) -> tuple:
    """Download weekly, daily, intraday frames for one ticker (Parquet-cached)."""
    weekly   = download_ticker(ticker, "1wk", PERIOD_WEEKLY)
    daily    = download_ticker(ticker, "1d",  PERIOD_DAILY)
    intraday = download_ticker(ticker, "15m", PERIOD_15M)
    return weekly, daily, intraday


def process_ticker(ticker: str) -> dict:
    """Full pipeline for one ticker: download → compute → chart → multi-model VLM.

    Downloads VIX + TV Derivative computation are handled here. The VIX chart
    is fetched from the module-level cache (downloaded once per pipeline run).
    """
    log.info("[%s] Fetching historical and intraday market data...", ticker)
    t0 = time.perf_counter()

    weekly, daily, intraday = _download_all_timeframes(ticker)

    wc, wv = to_f32_1d(weekly["Close"]),    to_f32_1d(weekly["Volume"])
    dc, dv = to_f32_1d(daily["Close"]),     to_f32_1d(daily["Volume"])
    ic, iv = to_f32_1d(intraday["Close"]),  to_f32_1d(intraday["Volume"])

    log.info("[%s] Computing rolling VWMA values & indicators...", ticker)
    vw_w = _vwma_loop(wc, wv, VWMA_WINDOW, _init_nan(wc.shape[0]))
    vw_d = _vwma_loop(dc, dv, VWMA_WINDOW, _init_nan(dc.shape[0]))
    vw_i = _vwma_loop(ic, iv, VWMA_WINDOW, _init_nan(ic.shape[0]))

    # Compute TV Derivative on tail(CHART_BARS) only (~12ms per call).
    # This keeps the TV solver fast and bounds total CPU cost.
    log.info("[%s] Computing TV-regularized trend slopes...", ticker)
    tv_w = _compute_tv_derivative(wc[-CHART_BARS:])
    tv_d = _compute_tv_derivative(dc[-CHART_BARS:])
    tv_i = _compute_tv_derivative(ic[-CHART_BARS:])

    # VIX metrics + shared chart path (downloaded once per pipeline run).
    vix_df = download_vix()
    vix_metrics = _compute_vix_metrics(vix_df)
    vix_chart_path = get_vix_chart_path()

    metrics = compute_metrics(dc, dv, vix_metrics=vix_metrics)

    log.info("[%s] Rendering technical analysis charts...", ticker)
    img_paths = generate_chart_images(
        weekly, daily, intraday, vw_w, vw_d, vw_i, ticker,
        tv_w=tv_w, tv_d=tv_d, tv_i=tv_i,
        vix_chart_path=vix_chart_path,
    )

    log.info("[%s] Querying multi-model VLM consensus (%s)...",
             ticker, " + ".join(VLM_PRIMARY_MODELS))
    forecast, disagreement, confidence = multi_model_consensus(img_paths, metrics)

    validate_forecast(forecast, metrics["close"], ticker)

    elapsed = time.perf_counter() - t0
    log.info("[%s] pipeline complete in %.1fs | confidence=%s | disagreement=%.2f%% | forecast=%s",
             ticker, elapsed, confidence, disagreement, forecast)

    return {
        "ticker":         ticker,
        "forecast":       forecast,
        "close":          metrics["close"],
        "daily":          daily,
        "disagreement":   disagreement,
        "confidence":     confidence,
        "vix_trend":      vix_metrics["vix_trend"],
    }


# =============================================================================
# Output formatting
# =============================================================================

def _fmt_num(v) -> str:
    """Format a single float value; return 'N/A' for None/NaN/Inf."""
    if v is None:
        return "N/A"
    try:
        fv = float(v)
        if not math.isfinite(fv):
            return "N/A"
        return f"{fv:.2f}".replace(".", ",")
    except (TypeError, ValueError):
        return "N/A"


def _fmt_pct(v) -> str:
    """Format a percentage; return 'N/A' for None/NaN/Inf."""
    if v is None:
        return "N/A"
    try:
        fv = float(v)
        if not math.isfinite(fv):
            return "N/A"
        return f"{fv:+.1f}%".replace(".", ",")
    except (TypeError, ValueError):
        return "N/A"


def _var_pct(r: dict) -> float:
    """Compute Var % (D+5 vs last close) for sorting.

    Returns -inf for any missing / NaN / Inf / non-positive inputs so that
    broken entries sort cleanly to the bottom of a descending list.
    """
    f = r.get("forecast")
    close_price = r.get("close", None)
    if not f or len(f) < 5 or close_price is None:
        return float("-inf")
    try:
        close_val = float(close_price)
        f_val     = float(f[4])
    except (TypeError, ValueError):
        return float("-inf")
    if not math.isfinite(close_val) or close_val <= 0:
        return float("-inf")
    if not math.isfinite(f_val):
        return float("-inf")
    return 100.0 * ((f_val / close_val) - 1.0)


def format_results_table(results: list) -> str:
    """Generate Markdown text aligning output sequences with Var % and Confidence."""
    cols = ["Ticker", "D+1", "D+2", "D+3", "D+4", "D+5", "Var %", "Conf."]
    rows = []
    for r in sorted(results, key=_var_pct, reverse=True):
        f   = r["forecast"]
        var = _var_pct(r)
        conf = r.get("confidence", "n/a")
        cells = [r["ticker"]] + [_fmt_num(v) for v in f] + [_fmt_pct(var), conf]
        rows.append("  ".join(f"{c:>10}" for c in cells))
    header = "  ".join(f"{c:>10}" for c in cols)
    sep    = "  ".join("-" * 10 for _ in cols)
    return "\n".join([header, sep] + rows)


# =============================================================================
# Main Plot Summary Matrix
# =============================================================================

def _style_plot_spines_and_legend(ax) -> None:
    """Apply dark-theme styling to plot spines and legend."""
    for spine in ax.spines.values():
        spine.set_color("#30363d")
    ax.legend(facecolor="#161b22", edgecolor="#30363d",
              labelcolor="#c9d1d9", loc="upper left", fontsize=8)


def _draw_single_ticker_forecast(ax, ticker: str, forecast: list, daily_df) -> None:
    """Draw the 45-day history + 5-day forecast dashed line for one ticker."""
    df_45 = daily_df.tail(45).copy()
    if not isinstance(df_45.index, pd.DatetimeIndex):
        df_45.index = pd.to_datetime(df_45.index)

    last_date  = df_45.index[-1]
    last_close = float(df_45["Close"].iloc[-1])

    future_dates   = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=5)
    forecast_dates = pd.DatetimeIndex([last_date]).append(future_dates)
    forecast_prices = [last_close] + list(forecast)

    ax.set_facecolor("#0d1117")
    ax.plot(df_45.index, df_45["Close"],
            color="#58a6ff", linewidth=1.8, label="Historical (45d)")
    ax.plot(forecast_dates, forecast_prices,
            color="#ff7b72", linewidth=1.8, linestyle="--",
            marker="o", markersize=4, label="Predicted (D+1 to D+5)")

    ax.set_title(f"{ticker} - Price & Forecast",
                 color="#c9d1d9", fontsize=12, fontweight="bold")
    ax.tick_params(colors="#8b949e", labelsize=9)
    ax.grid(True, color="#30363d", linestyle=":", alpha=0.6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

    _style_plot_spines_and_legend(ax)


def _draw_ticker_plot_error(ax, ticker: str, e: Exception) -> None:
    """Render an error placeholder panel when plotting a ticker fails."""
    ax.set_facecolor("#0d1117")
    ax.text(0.5, 0.5, f"Error plotting {ticker}:\n{str(e)}",
            color="#f85149", ha="center", va="center")
    ax.set_title(f"{ticker} - Plot Error", color="#f85149")


def _plot_single_matrix_ax(ax, ticker: str, forecast: list, daily_df) -> None:
    """Wrap single-ticker plotting with a try/except for resilience."""
    try:
        _draw_single_ticker_forecast(ax, ticker, forecast, daily_df)
    except Exception as e:
        _draw_ticker_plot_error(ax, ticker, e)


def _display_plot_matrix_ipython(out_path: str) -> None:
    """Display the summary PNG in a Colab/Jupyter notebook."""
    try:
        print("\n[Summary Plot] Final Forecast Summary Matrix:")
        ipy_display(ipy_Image(filename=out_path))
    except Exception as exc:
        log.debug("IPython display failed for final matrix: %s", exc)


def _display_plot_matrix(out_path: str) -> None:
    """Display the summary chart (IPython if available, else plt.show)."""
    if _HAS_IPY:
        _display_plot_matrix_ipython(out_path)
    else:
        plt.show()


def generate_forecast_matrix_chart(results: list) -> str:
    """Render the multi-ticker summary matrix chart to PNG."""
    num_tickers = len(results)
    if num_tickers == 0:
        return ""

    cols = 2
    rows = math.ceil(num_tickers / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(14, 4.5 * rows), facecolor="#0d1117")
    axes_flat = np.asarray(axes).ravel()

    for idx, r in enumerate(results):
        _plot_single_matrix_ax(axes_flat[idx], r["ticker"], r["forecast"], r["daily"])

    for unused_idx in range(num_tickers, len(axes_flat)):
        fig.delaxes(axes_flat[unused_idx])

    fig.autofmt_xdate()
    fig.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "vista_forecast_matrix.png")
    fig.savefig(out_path, dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")

    _display_plot_matrix(out_path)
    plt.close(fig)
    return out_path


# =============================================================================
# Main execution block
# =============================================================================

def _execute_ticker_pipelines(tickers: list) -> tuple:
    """Dispatch process_ticker across tickers via ThreadPoolExecutor."""
    n_workers = max(1, min(len(tickers), TICKER_PIPELINE_WORKERS))
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(process_ticker, t): t for t in tickers}
        return futures, list(as_completed(futures))


def _handle_ticker_future(fut, futures: dict, results: list, failed: list) -> None:
    """Collect one ticker's result; on failure record ticker in ``failed``."""
    ticker = futures[fut]
    try:
        results.append(fut.result())
    except Exception as exc:
        log.error("Pipeline failed for %s: %s", ticker, exc)
        failed.append(ticker)


def _collect_ticker_results(futures: dict, completed_futures, results: list, failed: list) -> None:
    """Drain all ticker futures into results / failed lists."""
    for fut in completed_futures:
        _handle_ticker_future(fut, futures, results, failed)


def _print_critical_error_message(exc: Exception) -> None:
    """Print a critical-error banner and terminate guidance."""
    print("\n" + "=" * 70)
    print("EXECUTION TERMINATED DUE TO CRITICAL ERROR")
    print("=" * 70)
    print(f"Error: {exc}")
    print("=" * 70)
    print("The pipeline cannot finalize output tables.")
    print("=" * 70)


def _print_final_results(results: list, model_label: str, failed: list = None) -> None:
    """Print the final Markdown-aligned results table."""
    print("\n" + "=" * 40 + " FORECAST TABULAR RESULTS " + "=" * 40)
    print(f"Models/Provider: NVIDIA NIM (dual-model: {model_label})")
    print("-" * 106)
    print(format_results_table(results))
    print("=" * 106)
    if failed:
        print(f"\n[WARNING] Failed tickers ({len(failed)}): {', '.join(failed)}")


def _validate_model_registry() -> None:
    """Verify that every primary model is present in MODEL_REGISTRY."""
    for m in VLM_PRIMARY_MODELS:
        if m not in MODEL_REGISTRY:
            print(f"ERROR: VLM_PRIMARY_MODELS references unknown model '{m}'.")
            print(f"Valid options: {', '.join(MODEL_REGISTRY.keys())}")
            sys.exit(1)


def main() -> None:
    """Orchestrate the pipeline: parse config, run tickers, render outputs."""
    tickers = TICKERS_INPUT.strip().split()
    if not tickers:
        print("No tickers provided. Set TICKERS_INPUT at the top of the script.")
        return

    _validate_model_registry()
    model_label = " + ".join(MODEL_REGISTRY[m]["label"] for m in VLM_PRIMARY_MODELS)

    log.info("VISTA pipeline started | Provider: NVIDIA NIM | Models: %s | Tickers: %s",
             model_label, tickers)

    # Pre-warm the VIX cache + chart once (downloads VIX, renders the shared
    # PNG, populates the module-level cache). All tickers then reuse the same
    # chart path. Failures here are non-fatal — process_ticker will gracefully
    # skip the VIX chart for every ticker.
    log.info("Pre-warming VIX cache (downloaded once, shared across all tickers)...")
    try:
        vix_path = get_vix_chart_path()
        if vix_path is not None:
            log.info("VIX cache ready: %s", vix_path)
        else:
            log.warning("VIX cache unavailable — pipeline will run without VIX charts.")
    except Exception as exc:
        log.warning("VIX pre-warm failed: %s — continuing without VIX.", exc)

    results = []
    failed  = []
    try:
        futures, completed = _execute_ticker_pipelines(tickers)
        _collect_ticker_results(futures, completed, results, failed)
    except Exception as exc:
        _print_critical_error_message(exc)
        sys.exit(1)

    if not results:
        print("\nNo successful results. Exiting.")
        if failed:
            print(f"Failed tickers: {', '.join(failed)}")
        sys.exit(1)

    _print_final_results(results, model_label, failed)

    try:
        generate_forecast_matrix_chart(results)
    except Exception as exc:
        log.error("Failed to generate final summary visualization matrix: %s", exc)


if __name__ == "__main__":
    main()
    try:
        from google.colab import runtime
        runtime.unassign()
    except ImportError:
        pass


15:19:34 - INFO - VISTA pipeline started | Provider: NVIDIA NIM | Models: MiniMax-M3 (minimaxai/minimax-m3) + Nemotron 3 Nano Omni 30B A3B (Reasoning) + Ministral 3 14B Instruct 2512 FP8 (mistralai/ministral-14b-instruct-2512) | Tickers: ['BTC-USD', 'SOL-USD', 'XRP-USD', 'QDFI11.SA', 'URPR11.SA', 'VSLH11.SA', 'BYND', 'CHTR', 'PATH', 'PYPL', 'UPST', 'AMAT', 'MU', 'STX']
15:19:34 - INFO - Pre-warming VIX cache (downloaded once, shared across all tickers)...
15:19:35 - INFO - VIX chart rendered once (shared across all tickers).
15:19:35 - INFO - VIX cache ready: /content/vista_vix_daily.png
15:19:35 - INFO - [BTC-USD] Fetching historical and intraday market data...
15:19:35 - INFO - [SOL-USD] Fetching historical and intraday market data...
15:19:35 - INFO - [XRP-USD] Fetching historical and intraday market data...
15:19:35 - INFO - [QDFI11.SA] Fetching historical and intraday market data...
15:19:37 - INFO - [BTC-USD] Computing rolling VWMA values & indicators...
15:19:37 - INFO - [SOL-US